
# **ML Modelling**

In [1]:
# LOAD DATA
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os, joblib

folders = glob.glob("/content/drive/MyDrive/Fre_data_after_corr_*")

if len(folders) == 0:
    raise ValueError("No saved dataset found. Check your folder name.")

latest_folder = max(folders, key=os.path.getmtime)
print("Loading data from:", latest_folder)

# FIX: correct filenames
X_train = joblib.load(os.path.join(latest_folder, "X_train_final.pkl"))
X_test = joblib.load(os.path.join(latest_folder, "X_test_final.pkl"))
y_train = joblib.load(os.path.join(latest_folder, "y_train.pkl"))
y_test = joblib.load(os.path.join(latest_folder, "y_test.pkl"))


# LOAD FEATURE SELECTION RESULTS
fs_folders = glob.glob("/content/drive/MyDrive/Fre_FS_results_*")

if len(fs_folders) == 0:
    raise ValueError("No FS results found.")

latest_fs_folder = max(fs_folders, key=os.path.getmtime)
print("Loading FS from:", latest_fs_folder)

fs_methods = {}

for file in os.listdir(latest_fs_folder):
    if file.endswith(".pkl"):
        name = file.replace(".pkl", "")
        fs_methods[name] = joblib.load(os.path.join(latest_fs_folder, file))

# ADD full dataset (no FS)
fs_methods["ALL_FEATURES"] = X_train.columns.tolist()


# SMOTE (TRAIN ONLY)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Import
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# EVALUATION METRICS FUNCTION
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision (W)": precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Recall (W)": recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "F1 (W)": f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Precision (Macro)": precision_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "Recall (Macro)": recall_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "F1 (Macro)": f1_score(y_test, y_pred, average='macro', zero_division=0) * 100,
    }

def print_results(results):
    for k, v in results.items():
        print(f"{k}: {v:.2f}%")

Mounted at /content/drive
Loading data from: /content/drive/MyDrive/Fre_data_after_corr_20260507_1542
Loading FS from: /content/drive/MyDrive/Fre_FS_results_20260507_1542


In [ ]:
# Print Loaded Feature Sets

print("\n================ FEATURE SETS LOADED ================")

for method, feats in fs_methods.items():
    print(f"\n{method}:")
    print(f"Number of features: {len(feats)}")
    print("Features:", feats)



================ FEATURE SETS LOADED ================

ANOVA:
Number of features: 15
Features: ['Weekly_Exercise_Duration', 'Risk Factor - Exercise', 'Year', 'Cooling Down', 'Balance in Sitting and Standing', 'Walking', 'Risk Factor - Smoking', 'Exercise Habit - Duration', 'Occupation', 'Predicted Risk Level Encoded', 'Gait', 'Risk Factor - ECHO - EF', 'Test Today - peak HR', 'Age', 'Family History']

Mutual_Info:
Number of features: 15
Features: ['Weekly_Exercise_Duration', 'Risk Factor - Exercise', 'Predicted Risk Level Encoded', 'Gait', 'Cooling Down', 'Exercise Habit - Mode', 'Marital Status', 'Test Today - peak HR', 'Risk Factor - Stress', 'Test Today - Termination Cause', 'Exercise Habit - Duration', 'Walking', 'Lives With', 'Functional Activity', 'Smoking']

RFE_LR:
Number of features: 15
Features: ['Exercise Habit - Duration', 'Weekly_Exercise_Duration', 'Predicted Risk Level Encoded', 'RISK  - Risk Type', 'Year', 'Gait', 'Risk Factor - Exercise', 'Posture', 'Risk Factor - DM'

In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# **Conventional Models**

*   Decision Tree
*   Logistic Regression (Classifier)
*   k-Nearest Neighbors (k-NN)



In [ ]:
# CONVENTIONAL MODELS

import numpy as np
import random
import os
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================================
# FIXED SEED
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================================
# SAFE CV
# =========================================================

min_class_count = min(Counter(y_train).values())

n_splits = min(5, min_class_count)
n_splits = max(2, n_splits)

print(f"Using {n_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# MODELS
# =========================================================

models = {

    "Decision Tree": (

        DecisionTreeClassifier(
            class_weight='balanced',
            random_state=SEED
        ),

        {
            "max_depth": list(range(2, 30))
        }
    ),

    "Logistic Regression": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("model", LogisticRegression(
                max_iter=5000,
                random_state=SEED
            ))
        ]),

        {
            "model__C": np.logspace(-3, 2, 10)
        }
    ),

    "k-NN": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("model", KNeighborsClassifier())
        ]),

        {
            "model__n_neighbors": list(range(3, 15))
        }
    )
}

# =========================================================
# LOOP
# =========================================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== CONVENTIONAL | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train.loc[:, features]
    X_test_sel = X_test.loc[:, features]

    for name, (model, params) in models.items():

        print(f"\n{name}")

        total_space = int(
            np.prod([len(v) for v in params.values()])
        )

        n_iter = min(10, total_space)

        search = RandomizedSearchCV(

            estimator=model,

            param_distributions=params,

            n_iter=n_iter,

            cv=cv,

            scoring='accuracy',

            n_jobs=1,

            random_state=SEED
        )

        # TRAIN
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)

        print_results(
            evaluate(y_test, y_pred)
        )

Using 5-Fold Stratified CV


========== CONVENTIONAL | FEATURE SET: ANOVA ==========

Decision Tree
Best Params: {'max_depth': 10}
Accuracy: 58.14%
Precision (W): 57.98%
Recall (W): 58.14%
F1 (W): 57.60%
Precision (Macro): 56.00%
Recall (Macro): 53.38%
F1 (Macro): 53.96%

Logistic Regression
Best Params: {'model__C': np.float64(100.0)}
Accuracy: 53.49%
Precision (W): 62.11%
Recall (W): 53.49%
F1 (W): 52.92%
Precision (Macro): 55.63%
Recall (Macro): 47.40%
F1 (Macro): 47.34%

k-NN
Best Params: {'model__n_neighbors': 14}
Accuracy: 20.93%
Precision (W): 32.77%
Recall (W): 20.93%
F1 (W): 21.74%
Precision (Macro): 32.05%
Recall (Macro): 24.86%
F1 (Macro): 23.77%


========== CONVENTIONAL | FEATURE SET: Mutual_Info ==========

Decision Tree
Best Params: {'max_depth': 27}
Accuracy: 60.47%
Precision (W): 61.54%
Recall (W): 60.47%
F1 (W): 60.52%
Precision (Macro): 55.15%
Recall (Macro): 53.38%
F1 (Macro): 53.73%

Logistic Regression
Best Params: {'model__C': np.float64(100.0)}
Accuracy: 58.14%


# **Ensemble models**

*   Random Forest
*   CatBoost
*   LightGBM


In [ ]:
# ENSEMBLE MODELS

import numpy as np
import random
import os

from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# INSTALL MISSING PACKAGES
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# CatBoost
try:
    from catboost import CatBoostClassifier
except ModuleNotFoundError:
    install("catboost")
    from catboost import CatBoostClassifier

# LightGBM
try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError:
    install("lightgbm")
    from lightgbm import LGBMClassifier

# XGBoost
try:
    import xgboost as xgb
except ModuleNotFoundError:
    install("xgboost")
    import xgboost as xgb

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# MODELS
# ============================================

ensembles = {

    "Random Forest": (
        RandomForestClassifier(
            class_weight='balanced',
            random_state=SEED,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
        }
    ),

    "LightGBM": (
        LGBMClassifier(
            class_weight='balanced',
            random_state=SEED,
            verbosity=-1,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
        }
    ),

    "XGBoost": (
        xgb.XGBClassifier(
            eval_metric='mlogloss',
            random_state=SEED,
            use_label_encoder=False,
            n_jobs=1
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1],
        }
    ),

    "CatBoost": (
        CatBoostClassifier(
            auto_class_weights="Balanced",
            verbose=0,
            random_seed=SEED
        ),
        {
            "iterations": [200, 400],
            "depth": [4, 6],
            "learning_rate": [0.05, 0.1],
        }
    ),
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== ENSEMBLE | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    for name, (model, params) in ensembles.items():

        print(f"\n{name}")

        total_space = int(np.prod([len(v) for v in params.values()]))
        n_iter = min(10, total_space)

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=n_iter,
            cv=cv,
            scoring='accuracy',
            n_jobs=1,
            random_state=SEED
        )

        search.fit(X_train_sel, y_train)

        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)
        print_results(evaluate(y_test, y_pred))

Using 5-Fold Stratified CV


========== ENSEMBLE | FEATURE SET: ANOVA ==========

Random Forest
Best Params: {'n_estimators': 200, 'max_depth': 10}
Accuracy: 62.79%
Precision (W): 66.61%
Recall (W): 62.79%
F1 (W): 61.01%
Precision (Macro): 68.77%
Recall (Macro): 58.05%
F1 (Macro): 57.52%

LightGBM
Best Params: {'n_estimators': 200, 'learning_rate': 0.05}
Accuracy: 62.79%
Precision (W): 64.41%
Recall (W): 62.79%
F1 (W): 61.68%
Precision (Macro): 59.24%
Recall (Macro): 59.38%
F1 (Macro): 56.66%

XGBoost
Best Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 76.74%
Precision (W): 80.61%
Recall (W): 76.74%
F1 (W): 76.54%
Precision (Macro): 79.55%
Recall (Macro): 73.88%
F1 (Macro): 73.39%

CatBoost
Best Params: {'learning_rate': 0.1, 'iterations': 400, 'depth': 4}
Accuracy: 74.42%
Precision (W): 78.25%
Recall (W): 74.42%
F1 (W): 74.43%
Precision (Macro): 77.19%
Recall (Macro): 70.24%
F1 (Macro): 70.70%


========== ENSEMBLE | FEATURE SET: Mutual_Info ==========


# **Deep learning**

*   FCN
*   BPNN
*   CNN








In [ ]:
# DEEP LEARNING MODELS

import numpy as np
import tensorflow as tf
import random
import os

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# FIXED SEED
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

os.environ['TF_DETERMINISTIC_OPS'] = '1'

# SETTINGS
EPOCHS = 40
BATCH_SIZE = 32

# CLASS WEIGHTS
def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

# MODEL DEFINITIONS
def build_fcn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_bpnn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_cnn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Reshape((input_dim, 1)),
        tf.keras.layers.Conv1D(16, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# AUTOENCODER + CLASSIFIER
def build_autoencoder_classifier(input_dim, num_classes):

    inputs = tf.keras.Input(shape=(input_dim,))

    encoded = tf.keras.layers.Dense(64, activation='relu')(inputs)
    encoded = tf.keras.layers.Dense(32, activation='relu')(encoded)

    decoded = tf.keras.layers.Dense(64, activation='relu')(encoded)
    decoded = tf.keras.layers.Dense(input_dim, activation='linear', name="reconstruction")(decoded)

    classifier = tf.keras.layers.Dense(num_classes, activation='softmax', name="classification")(encoded)

    model = tf.keras.Model(inputs=inputs, outputs=[decoded, classifier])

    return model

# TRAINING FUNCTIONS
def train_standard(model_fn, X_train, y_train, X_test, y_test):

    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]

    model = model_fn(input_dim, num_classes)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    class_weights = get_class_weights(y_train)

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        class_weight=class_weights,
        shuffle=True
    )

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    return evaluate(y_test, y_pred)

def train_autoencoder(X_train, y_train, X_test, y_test):

    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]

    model = build_autoencoder_classifier(input_dim, num_classes)

    model.compile(
        optimizer='adam',
        loss={
            "reconstruction": "mse",
            "classification": "sparse_categorical_crossentropy"
        },
        loss_weights={
            "reconstruction": 0.3,
            "classification": 1.0
        },
        metrics={"classification": "accuracy"}
    )

    model.fit(
        X_train,
        {"reconstruction": X_train, "classification": y_train},
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        shuffle=True
    )

    preds = model.predict(X_test, verbose=0)[1]
    y_pred = np.argmax(preds, axis=1)

    return evaluate(y_test, y_pred)

# MAIN LOOP
for fs_name, features in fs_methods.items():

    print(f"\n\n========== DEEP LEARNING | FEATURE SET: {fs_name} ==========")

    # SCALE
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train[features])
    X_test_s = scaler.transform(X_test[features])

    # FCN
    print("\nFCN")
    print_results(train_standard(build_fcn, X_train_s, y_train, X_test_s, y_test))

    # BPNN
    print("\nBPNN")
    print_results(train_standard(build_bpnn, X_train_s, y_train, X_test_s, y_test))

    # CNN
    print("\nCNN")
    print_results(train_standard(build_cnn, X_train_s, y_train, X_test_s, y_test))

    # AUTOENCODER
    print("\nAutoencoder Classifier")
    print_results(train_autoencoder(X_train_s, y_train, X_test_s, y_test))



========== DEEP LEARNING | FEATURE SET: ANOVA ==========

FCN
Accuracy: 48.84%
Precision (W): 49.43%
Recall (W): 48.84%
F1 (W): 49.01%
Precision (Macro): 44.13%
Recall (Macro): 44.45%
F1 (Macro): 44.13%

BPNN
Accuracy: 46.51%
Precision (W): 45.14%
Recall (W): 46.51%
F1 (W): 45.28%
Precision (Macro): 40.18%
Recall (Macro): 39.88%
F1 (Macro): 39.58%

CNN


Accuracy: 48.84%
Precision (W): 49.42%
Recall (W): 48.84%
F1 (W): 48.95%
Precision (Macro): 42.21%
Recall (Macro): 42.38%
F1 (Macro): 42.10%

Autoencoder Classifier
Accuracy: 46.51%
Precision (W): 42.42%
Recall (W): 46.51%
F1 (W): 43.52%
Precision (Macro): 36.16%
Recall (Macro): 37.62%
F1 (Macro): 35.89%


========== DEEP LEARNING | FEATURE SET: Mutual_Info ==========

FCN
Accuracy: 44.19%
Precision (W): 43.98%
Recall (W): 44.19%
F1 (W): 43.61%
Precision (Macro): 38.77%
Recall (Macro): 37.12%
F1 (Macro): 37.30%

BPNN
Accuracy: 51.16%
Precision (W): 50.81%
Recall (W): 51.16%
F1 (W): 50.36%
Precision (Macro): 47.69%
Recall (Macro): 44.95%
F1 (Macro): 45.49%

CNN
Accuracy: 46.51%
Precision (W): 41.48%
Recall (W): 46.51%
F1 (W): 43.72%
Precision (Macro): 34.72%
Recall (Macro): 38.21%
F1 (Macro): 36.27%

Autoencoder Classifier
Accuracy: 46.51%
Precision (W): 40.67%
Recall (W): 46.51%
F1 (W): 41.09%
Precision (Macro): 33.76%
Recall (Macro): 35.79%
F1 (Macro): 32.49%


========== DEEP LEARNIN

# **Transformer models**

*   FT-Transformer
*   TabTransformer
*   HistGB (Attention-) like



In [ ]:
# TRANSFORMER MODELS

import numpy as np
import random
import os
import warnings

warnings.filterwarnings("ignore")

from collections import Counter

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.ensemble import HistGradientBoostingClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================================
# FIXED SEED
# =========================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================================
# SAFE CV
# =========================================================

min_class_count = min(Counter(y_train).values())

n_splits = min(5, min_class_count)
n_splits = max(2, n_splits)

print(f"Using {n_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================================
# MODELS
# =========================================================

transformer_models = {

    # FT-Transformer Approximation
    "FT-Transformer (Approx)": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("mlp", MLPClassifier(
                hidden_layer_sizes=(256, 128, 64),
                max_iter=1500,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=SEED
            ))
        ]),

        {
            "mlp__alpha": [0.0001, 0.001]
        }
    ),

    # TabTransformer Approximation
    "TabTransformer (Approx)": (

        ImbPipeline([

            ("ros", RandomOverSampler(random_state=SEED)),

            ("scaler", StandardScaler()),

            ("mlp", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                max_iter=1200,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=SEED
            ))
        ]),

        {
            "mlp__alpha": [0.0001, 0.001]
        }
    ),
}

# =========================================================
# LOOP
# =========================================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== TRANSFORMER | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train.loc[:, features]
    X_test_sel = X_test.loc[:, features]

    for name, (model, params) in transformer_models.items():

        print(f"\n{name}")

        total_space = int(
            np.prod([len(v) for v in params.values()])
        )

        n_iter = min(10, total_space)

        search = RandomizedSearchCV(

            estimator=model,

            param_distributions=params,

            n_iter=n_iter,

            cv=cv,

            scoring='accuracy',

            n_jobs=1,

            random_state=SEED
        )

        # TRAIN
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)

        print_results(
            evaluate(y_test, y_pred)
        )

Using 5-Fold Stratified CV


========== TRANSFORMER | FEATURE SET: ANOVA ==========

FT-Transformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 27.91%
Precision (W): 33.59%
Recall (W): 27.91%
F1 (W): 26.69%
Precision (Macro): 32.30%
Recall (Macro): 29.19%
F1 (Macro): 26.65%

TabTransformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 18.60%
Precision (W): 23.02%
Recall (W): 18.60%
F1 (W): 19.51%
Precision (Macro): 19.67%
Recall (Macro): 19.05%
F1 (Macro): 18.27%

Attention-like Boosting (HistGB)
Best Params: {'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 65.12%
Precision (W): 58.99%
Recall (W): 65.12%
F1 (W): 61.32%
Precision (Macro): 51.07%
Recall (Macro): 59.38%
F1 (Macro): 54.31%


========== TRANSFORMER | FEATURE SET: Mutual_Info ==========

FT-Transformer (Approx)
Best Params: {'mlp__alpha': 0.0001}
Accuracy: 32.56%
Precision (W): 32.35%
Recall (W): 32.56%
F1 (W): 32.16%
Precision (Macro): 26.61%
Recall (Macro): 27.90%
F1 (Macro): 26.99%

TabTransformer (Ap

In [4]:
# TABNET MODEL

import numpy as np
import pandas as pd
import random
import os
import warnings

from collections import Counter

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

# ============================================
# SUPPRESS TABNET WARNING
# ============================================

warnings.filterwarnings(
    "ignore",
    message="No early stopping will be performed"
)

# ============================================
# INSTALL TABNET IF MISSING
# ============================================

import sys
import subprocess

def install(package):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", package]
    )

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ModuleNotFoundError:
    install("pytorch-tabnet")
    from pytorch_tabnet.tab_model import TabNetClassifier

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# TABNET WRAPPER
# ============================================

class TabNetWrapper(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_d=16,
        n_a=16,
        n_steps=3,
        gamma=1.3,
        lambda_sparse=1e-4
    ):

        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse

    def fit(self, X, y):

        # Convert dataframe to numpy
        X_np = X.values if hasattr(X, "values") else X

        # Internal validation split
        X_train_inner, X_valid_inner, y_train_inner, y_valid_inner = train_test_split(
            X_np,
            y,
            test_size=0.1,
            stratify=y,
            random_state=SEED
        )

        # Create model
        self.model_ = TabNetClassifier(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            seed=SEED,
            verbose=0
        )

        # Train with early stopping
        self.model_.fit(
            X_train=X_train_inner,
            y_train=y_train_inner,

            eval_set=[
                (X_valid_inner, y_valid_inner)
            ],

            eval_metric=["accuracy"],

            max_epochs=100,
            patience=20,

            batch_size=32,
            virtual_batch_size=16
        )

        return self

    def predict(self, X):

        X_np = X.values if hasattr(X, "values") else X

        return self.model_.predict(X_np)

    def predict_proba(self, X):

        X_np = X.values if hasattr(X, "values") else X

        return self.model_.predict_proba(X_np)

# ============================================
# PIPELINE
# ============================================

tabnet_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", TabNetWrapper())
])

# ============================================
# PARAMETER SPACE
# ============================================

param_dist = {

    "model__n_d": [8, 16, 32],

    "model__n_a": [8, 16, 32],

    "model__n_steps": [3, 5],

    "model__gamma": [1.0, 1.3],

    "model__lambda_sparse": [1e-3, 1e-4]
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== TABNET | FEATURE SET: {fs_name} ==========")

    # Feature selection
    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # Prevent n_iter warning
    total_space = int(
        np.prod([len(v) for v in param_dist.values()])
    )

    n_iter = min(10, total_space)

    # Randomized Search
    search = RandomizedSearchCV(
        estimator=tabnet_pipeline,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,
        random_state=SEED
    )

    # TRAIN
    search.fit(X_train_sel, y_train)

    # TEST
    y_pred = search.predict(X_test_sel)

    # RESULTS
    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 5-Fold Stratified CV


========== TABNET | FEATURE SET: ANOVA ==========

Early stopping occurred at epoch 27 with best_epoch = 7 and best_val_0_accuracy = 0.42857


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 30 with best_epoch = 10 and best_val_0_accuracy = 0.42857


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 58 with best_epoch = 38 and best_val_0_accuracy = 0.57143


...

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 27 with best_epoch = 7 and best_val_0_accuracy = 0.42857


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 38 with best_epoch = 18 and best_val_0_accuracy = 0.41176
Best Params: {'model__n_steps': 3, 'model__n_d': 8, 'model__n_a': 8, 'model__lambda_sparse': 0.001, 'model__gamma': 1.0}
Accuracy: 27.91%
Precision (W): 22.20%
Recall (W): 27.91%
F1 (W): 20.02%
Precision (Macro): 18.74%
Recall (Macro): 18.79%
F1 (Macro): 14.69%


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
